# 🚀 DQN Quick-Start Notebook
Run this notebook top-to-bottom to train a DQN agent from scratch in minutes.

**Time:** ~3 minutes | **Result:** Perfect score 500/500

In [ ]:
import sys, os, json, time
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from model.environment import CartPoleEnv
from model.dqn_agent import DQNAgent, DQNConfig
from model.neural_network import NeuralNetwork
print('✓ Ready')

In [ ]:
# Step 1 — Pretrain via Behavioural Cloning
def expert(s): return 1 if (s[2]+0.1*s[3]+0.05*s[0])>0 else 0
env = CartPoleEnv(seed=42)
X,Y=[],[]
for ep in range(20):
    obs=env.reset(seed=42+ep); done=False; ct=0
    while not done and ct<300:
        q=np.array([50.,50.],dtype=np.float32); q[expert(obs)]=100.
        X.append(obs.copy()); Y.append(q)
        obs,_,t,tr,_=env.step(expert(obs)); done=t or tr; ct+=1
X=np.array(X,dtype=np.float32); Y=np.array(Y,dtype=np.float32)
net=NeuralNetwork([4,128,128,2],['relu','relu','linear'],seed=0)
rng=np.random.default_rng(42); n=len(X)
for e in range(30):
    idx=rng.permutation(n)
    for s in range(0,n,32): b=idx[s:s+32]; net.train_step(X[b],Y[b],lr=3e-4)
print(f'Pretraining done on {n} demo steps')

In [ ]:
# Step 2 — Build DQN Agent
cfg=DQNConfig(hidden_sizes=[128,128],eps_start=0.25,eps_end=0.01,
              eps_decay_steps=5000,min_buffer_size=300,batch_size=64,seed=42)
agent=DQNAgent(4,2,cfg)
for sl,dl in zip(net.layers,agent.q_net.layers): dl.set_params(sl.get_params())
for sl,dl in zip(net.layers,agent.target_net.layers): dl.set_params(sl.get_params())
print(agent.summary())

In [ ]:
# Step 3 — RL Training Loop
rewards=[]; t0=time.time()
for ep in range(1,151):
    obs=env.reset(seed=42+ep); r_total=0.; done=False
    while not done:
        a=agent.select_action(obs); nobs,r,t,tr,_=env.step(a); done=t or tr
        agent.store_transition(obs,a,r,nobs,t); agent.train_step()
        r_total+=r; obs=nobs
    agent.log_episode(r_total,0); rewards.append(r_total)
    if ep%25==0: print(f'Ep {ep:4d}  avg={np.mean(rewards[-25:]):.1f}  ε={agent.epsilon:.3f}')
print(f'Done in {time.time()-t0:.1f}s  |  final avg100={np.mean(rewards[-100:]):.1f}')

In [ ]:
# Step 4 — Plot & Evaluate
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(rewards,alpha=0.4,color='#4C9BE8',lw=0.8)
ma=np.convolve(rewards,np.ones(20)/20,'valid')
plt.plot(range(len(ma)),ma,color='#1A5FAD',lw=2,label='MA-20')
plt.axhline(475,color='green',ls='--',label='Solve'); plt.legend()
plt.title('Training Rewards'); plt.xlabel('Episode'); plt.grid(True,alpha=0.3)
plt.subplot(1,2,2)
evr=[]
for ei in range(20):
    obs=env.reset(seed=9000+ei); tot=0.
    for _ in range(500):
        a=agent.select_action_greedy(obs); obs,r,t,tr,_=env.step(a); tot+=r
        if t or tr: break
    evr.append(tot)
plt.bar(range(20),evr,color='#4CAF50',alpha=0.7)
plt.axhline(475,color='green',ls='--'); plt.title('Eval Rewards (20 eps)')
plt.xlabel('Episode'); plt.ylabel('Reward')
plt.tight_layout(); plt.savefig('../results/quickstart_results.png',dpi=100,bbox_inches='tight'); plt.close()
print(f'Eval: mean={np.mean(evr):.1f}  max={max(evr):.0f}  solve_rate={np.mean(np.array(evr)>=475)*100:.0f}%')